# Landslide source and runout zones step 04: charts (minimum_scenario)

Runs the original landslide chart workflow against the source-and-runout zone post-processed summaries from step 03.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
output_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario_source_and_runout_zones'
damage_estimates_path = output_path / 'damage_estimates'

sector_summary_file = damage_estimates_path / 'sector_scenario_return_period_damages.csv'
sector_subsector_summary_file = damage_estimates_path / 'sector_subsector_scenario_return_period_damages.csv'

print('Output path:', output_path)
print('Sector summary file:', sector_summary_file)
print('Sector+subsector summary file:', sector_subsector_summary_file)


In [ ]:
if not sector_summary_file.exists():
    raise FileNotFoundError(
        f'Missing summary file: {sector_summary_file}. Run landslide_03_postprocess_summaries first.'
    )

sector_summary = pd.read_csv(sector_summary_file)

required_columns = {'Sector', 'Scenario', 'ReturnPeriod', 'Direct_Damages_USD'}
missing_columns = required_columns.difference(sector_summary.columns)
if missing_columns:
    raise KeyError(f'Missing required columns in {sector_summary_file.name}: {sorted(missing_columns)}')

sector_summary = sector_summary.copy()
sector_summary['ReturnPeriod'] = pd.to_numeric(sector_summary['ReturnPeriod'], errors='coerce').astype(int)
sector_summary['Direct_Damages_USD'] = pd.to_numeric(sector_summary['Direct_Damages_USD'], errors='coerce').fillna(0.0)

print('Rows:', len(sector_summary))
sector_summary.head(20)


In [ ]:
# Total USD direct damages by scenario and return period (across all sectors)
scenario_order = ['baseline', 'deforestation', 'reafforestation']

total_summary = (
    sector_summary
    .groupby(['Scenario', 'ReturnPeriod'], as_index=False)['Direct_Damages_USD']
    .sum()
)

total_summary['Scenario'] = pd.Categorical(total_summary['Scenario'], categories=scenario_order, ordered=True)
total_summary = total_summary.sort_values(['ReturnPeriod', 'Scenario']).reset_index(drop=True)

pivot_total = (
    total_summary
    .pivot_table(index='ReturnPeriod', columns='Scenario', values='Direct_Damages_USD', aggfunc='sum', fill_value=0.0)
    .reindex(columns=scenario_order)
    .fillna(0.0)
)

pivot_total


In [ ]:
# Grouped bar chart: direct damages (USD) by return period and scenario
return_periods = pivot_total.index.to_list()
bar_positions = np.arange(len(return_periods))
bar_width = 0.24

scenario_colors = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reafforestation': '#54A24B',
}

fig, ax = plt.subplots(figsize=(10, 6))

for idx, scenario in enumerate(scenario_order):
    if scenario not in pivot_total.columns:
        continue

    offsets = bar_positions + (idx - 1) * bar_width
    values = pivot_total[scenario].to_numpy()

    ax.bar(
        offsets,
        values,
        width=bar_width,
        label=scenario.capitalize(),
        color=scenario_colors.get(scenario, '#888888'),
        edgecolor='white',
        linewidth=0.8,
    )

ax.set_xticks(bar_positions)
ax.set_xticklabels([str(rp) for rp in return_periods])
ax.set_xlabel('Return period (years)')
ax.set_ylabel('Direct damages (USD)')
ax.set_title('Landslide Source and Runout Zone Direct Damages in USD by Return Period and Scenario')
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Scenario', frameon=True)

ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f'${value:,.0f}'))

plt.tight_layout()

chart_file = damage_estimates_path / 'landslide_source_and_runout_direct_damages_usd_by_return_period_and_scenario.png'
fig.savefig(chart_file, dpi=300, bbox_inches='tight')
print('Saved chart:', chart_file)
plt.show()


In [ ]:
# Optional: sector-by-sector USD table for interpretation
sector_pivot = (
    sector_summary
    .pivot_table(
        index=['Sector', 'ReturnPeriod'],
        columns='Scenario',
        values='Direct_Damages_USD',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reset_index()
)

for col in ['baseline', 'deforestation', 'reafforestation']:
    if col not in sector_pivot.columns:
        sector_pivot[col] = 0.0

sector_pivot['Deforestation_Change_USD'] = sector_pivot['deforestation'] - sector_pivot['baseline']
sector_pivot['Reafforestation_Change_USD'] = sector_pivot['reafforestation'] - sector_pivot['baseline']

sector_pivot = sector_pivot.sort_values(['Sector', 'ReturnPeriod']).reset_index(drop=True)

sector_table_file = damage_estimates_path / 'sector_return_period_scenario_damages_usd.csv'
sector_pivot.to_csv(sector_table_file, index=False)
print('Saved table:', sector_table_file)

sector_pivot.head(30)


In [ ]:
# RP-sector charts: direct damages and avoided damages
sector_order = ['buildings', 'transport', 'water', 'energy']
existing_sectors = [s for s in sector_order if s in sector_pivot['Sector'].unique()]
remaining_sectors = sorted([s for s in sector_pivot['Sector'].unique() if s not in existing_sectors])
plot_sectors = existing_sectors + remaining_sectors

if not plot_sectors:
    raise ValueError('No sectors found in sector_pivot for plotting.')

# Ensure deterministic RP order
rps = sorted(sector_pivot['ReturnPeriod'].dropna().unique().tolist())

scenario_styles = {
    'baseline': {'color': '#4C78A8', 'label': 'Baseline'},
    'deforestation': {'color': '#F58518', 'label': 'Deforestation'},
    'reafforestation': {'color': '#54A24B', 'label': 'Reafforestation'},
}

# 1) Direct damages by RP across sectors
n = len(plot_sectors)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for i, sector in enumerate(plot_sectors):
    ax = axes[i]
    d = sector_pivot[sector_pivot['Sector'] == sector].sort_values('ReturnPeriod')

    for scenario in ['baseline', 'deforestation', 'reafforestation']:
        if scenario not in d.columns:
            continue
        ax.plot(
            d['ReturnPeriod'],
            d[scenario],
            marker='o',
            linewidth=2,
            markersize=4,
            color=scenario_styles[scenario]['color'],
            label=scenario_styles[scenario]['label'],
        )

    ax.set_title(sector.capitalize())
    ax.grid(alpha=0.25)
    ax.set_xticks(rps)
    ax.set_xlabel('Return period (years)')
    ax.set_ylabel('Direct damages (USD)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f'${value/1e6:,.0f}m'))

# Hide unused axes
for j in range(n, len(axes)):
    axes[j].set_visible(False)

# Single shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Landslide Source and Runout Zone Direct Damages by Return Period Across Sectors', y=1.05)
plt.tight_layout()

direct_sector_chart = damage_estimates_path / 'landslide_source_and_runout_sector_direct_damages_by_rp_and_scenario.png'
fig.savefig(direct_sector_chart, dpi=300, bbox_inches='tight')
print('Saved chart:', direct_sector_chart)
plt.show()

# 2) Avoided damages by RP across sectors
sector_avoided_rp = sector_pivot.copy()
sector_avoided_rp['Avoided_EAD_Protection_USD'] = sector_avoided_rp['deforestation'] - sector_avoided_rp['baseline']
sector_avoided_rp['Avoided_EAD_Reafforestation_USD'] = sector_avoided_rp['baseline'] - sector_avoided_rp['reafforestation']
sector_avoided_rp['Combined_Benefit_USD'] = sector_avoided_rp['deforestation'] - sector_avoided_rp['reafforestation']

avoided_table_file = damage_estimates_path / 'sector_return_period_avoided_damages_usd.csv'
sector_avoided_rp.to_csv(avoided_table_file, index=False)
print('Saved table:', avoided_table_file)

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for i, sector in enumerate(plot_sectors):
    ax = axes[i]
    d = sector_avoided_rp[sector_avoided_rp['Sector'] == sector].sort_values('ReturnPeriod')

    ax.plot(
        d['ReturnPeriod'],
        d['Avoided_EAD_Reafforestation_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#2E7D32',
        label='Avoided via reafforestation (Baseline - Reafforestation)',
    )
    ax.plot(
        d['ReturnPeriod'],
        d['Avoided_EAD_Protection_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#EF6C00',
        label='Avoided via forest protection (Deforestation - Baseline)',
    )

    ax.axhline(0.0, color='#777777', linewidth=1, linestyle='--')
    ax.set_title(sector.capitalize())
    ax.grid(alpha=0.25)
    ax.set_xticks(rps)
    ax.set_xlabel('Return period (years)')
    ax.set_ylabel('Avoided damages (USD)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f'${value/1e6:,.0f}m'))

for j in range(n, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=1, frameon=True, bbox_to_anchor=(0.5, 1.06))
fig.suptitle('Landslide Source and Runout Zone Avoided Damages by Return Period Across Sectors', y=1.10)
plt.tight_layout()

avoided_sector_chart = damage_estimates_path / 'landslide_source_and_runout_sector_avoided_damages_by_rp.png'
fig.savefig(avoided_sector_chart, dpi=300, bbox_inches='tight')
print('Saved chart:', avoided_sector_chart)
plt.show()

sector_avoided_rp[['Sector','ReturnPeriod','Avoided_EAD_Reafforestation_USD','Avoided_EAD_Protection_USD','Combined_Benefit_USD']].head(20)